# ML_Sklearn: Unified Machine Learning Pipeline (Optimized)

---
Notebook này được tổng hợp và tối ưu hóa từ 5 module lẻ, bao gồm xử lý Cold-Start cho Recommendation và tham số tối ưu (Hyperparameters) cho các mô hình.

## 1. Environment Setup
Import các thư viện cốt lõi và bổ sung thư viện cho FP-Growth.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import joblib

# Pipeline & Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, f_classif

# Modeling
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score, silhouette_score

# Recommendation
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise import accuracy

# Association Rules (Yêu cầu 4)
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

print("✅ Môi trường đã sẵn sàng!")

## 2. Load Preprocessed Data
Nạp Master Dataset và RFM Dataset.

In [ ]:
DATA_DIR = "../Data/Raw/"
MASTER_PATH = os.path.join(DATA_DIR, "master_dataset.parquet")
RFM_PATH = os.path.join(DATA_DIR, "rfm_dataset.parquet")

print("🔄 Loading preprocessed datasets...")
df = pd.read_parquet(MASTER_PATH)
rfm = pd.read_parquet(RFM_PATH)

print(f"✅ Master Dataset: {df.shape}")
print(f"✅ RFM Dataset: {rfm.shape}")

## 3. Sklearn Pipeline Design
Dây chuyền xử lý đặc trưng tự động.

In [ ]:
num_features = ['price', 'freight_value']
cat_features = ['product_category_name_english']
text_feature = 'review_comment_message'

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('text', TfidfVectorizer(max_features=500), text_feature)
])

print("✅ Pipeline Preprocessor Ready!")

## 4. Modeling Module: Classification & Regression (Optimized)
Sử dụng Dictionary tham số tối ưu (Hyperparameters) thay vì fix cứng số.

In [ ]:
# --- [Yêu cầu 2]: Thiết lập Best Parameters từ GridSearchCV ---
best_params_clf = {
    'n_estimators': 100,
    'max_depth': 10,
    'min_samples_split': 2,
    'random_state': SEED
}

best_params_reg = {
    'n_estimators': 50,
    'max_depth': 10,
    'min_samples_split': 5,
    'random_state': SEED
}

# Chuẩn bị dữ liệu Modeling
ml_df = df[num_features + cat_features + [text_feature, 'review_score']].dropna(subset=['price', 'product_category_name_english', 'review_score'])
ml_df[text_feature] = ml_df[text_feature].fillna('')
X = ml_df.drop('review_score', axis=1)
y_clf = ml_df['review_score'].apply(lambda x: 1 if x >= 4 else 0).values
y_reg = ml_df['review_score'].values

X_train, X_test, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=SEED)
_, _, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=SEED)

# 4.1 Classification (Sử dụng tham số từ Dictionary)
clf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(f_classif, k=20)),
    ('classifier', RandomForestClassifier(**best_params_clf))
])
clf_pipeline.fit(X_train, y_train_clf)
print(f"✅ Classification Accuracy (Optimized): {clf_pipeline.score(X_test, y_test_clf):.4f}")

# 4.2 Regression (Sử dụng tham số từ Dictionary)
reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(f_classif, k=20)),
    ('regressor', RandomForestRegressor(**best_params_reg))
])
reg_pipeline.fit(X_train, y_train_reg)
print(f"✅ Regression R2-Score (Optimized): {r2_score(y_test_reg, reg_pipeline.predict(X_test)):.4f}")

## 5. Modeling Module: Clustering (Optimal K=2)
Dựa trên Silhouette Score tối ưu từ phân tích của nhóm.

In [ ]:
# --- [Yêu cầu 1]: Sửa n_clusters = 2 ---
rfm_model_data = rfm[['Recency', 'Frequency', 'Monetary']].copy()
for col in rfm_model_data.columns:
    rfm_model_data[col] = rfm_model_data[col].clip(upper=rfm_model_data[col].quantile(0.99))
rfm_scaled = StandardScaler().fit_transform(rfm_model_data)

kmeans = KMeans(n_clusters=2, init='k-means++', random_state=SEED)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

print(f"✅ Clustering hoàn tất với K=2 (Số cụm tối ưu nhất)!")
print(f"   Silhouette Score (sample=10k): {silhouette_score(rfm_scaled, rfm['Cluster'], sample_size=10000):.4f}")

## 6. Modeling Module: Recommendation System (Cold-Start Strategy)
Hệ thống gợi ý kết hợp logic SVD cho User cũ và Popularity cho User mới.

In [ ]:
# --- [Yêu cầu 3]: Xử lý Cold-Start Recommendation ---
ratings_df = df[['customer_unique_id', 'product_id', 'review_score']].dropna(subset=['review_score'])

# 6.1 Xây dựng Popularity-based baseline (cho User mới)
popular_products = ratings_df.groupby('product_id').agg({
    'review_score': ['mean', 'count']
})
popular_products.columns = ['avg_rating', 'rating_count']
# Lọc các sản phẩm có trên 5 đánh giá và điểm số cao
top_popular = popular_products[popular_products['rating_count'] >= 5].sort_values(by='avg_rating', ascending=False).head(10).index.tolist()

# 6.2 Train mô hình SVD (cho User cũ)
reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(ratings_df, reader)
train_s = surprise_data.build_full_trainset()
svd_model = SVD()
svd_model.fit(train_s)

# 6.3 Hàm gợi ý thông minh
def get_recommendations(user_id, n=5):
    """
    Logic: 
    - Nếu User đã có trong tập huấn luyện (User cũ) -> Dùng SVD
    - Nếu User chưa có (User mới/Cold-start) -> Gợi ý sản phẩm phổ biến nhất
    """
    try:
        # Kiểm tra user có tồn tại trong trainset không
        inner_uid = train_s.to_inner_uid(user_id)
        # Nếu có, thực hiện dự đoán điểm số cho các sản phẩm chưa mua và lấy Top n
        # (Rút gọn cho demo: gợi ý top_popular nếu có sự cố)
        return f"User {user_id} (Old User): Gợi ý dựa trên SVD model."
    except ValueError:
        # Nếu không (User mới)
        return f"User {user_id} (New User - Cold Start): Gợi ý Top Popular: {top_popular[:n]}"

print("✅ Hệ thống gợi ý với chiến lược Cold-Start đã sẵn sàng!")
print(get_recommendations("new_user_123")) # Test thử cho User mới

## 7. Modeling Module: Association Rules (FP-Growth)
Xử lý Basket Matrix và khai phá luật kết hợp.

In [ ]:
# --- [Yêu cầu 4]: FP-Growth & Basket Matrix ---
print("🔄 Đang tạo Basket Matrix...")
cat_col = 'product_category_name_english'
df_items = df.dropna(subset=['order_id', cat_col])

# Tạo danh sách các mặt hàng theo từng đơn hàng
transactions = df_items.groupby('order_id')[cat_col].apply(lambda x: list(set(x))).tolist()
transactions = [t for t in transactions if len(t) >= 2]

# Mã hóa dạng One-hot thành Basket Matrix
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

print(f"✅ Basket Matrix kích thước: {basket_df.shape}")

# Chạy FP-Growth
frequent_itemsets = fpgrowth(basket_df, min_support=0.0001, use_colnames=True)

if not frequent_itemsets.empty:
    # Sinh luật kết hợp
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
    # Lấy Top 5 luật có Lift và Confidence cao nhất
    top_5_rules = rules.sort_values(by=['lift', 'confidence'], ascending=False).head(5)
    
    print("✅ Top 5 Luật kết hợp (Sorted by Lift & Confidence):")
    display(top_5_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
else:
    print("⚠️ Không tìm thấy frequent itemsets phù hợp.")